# Urban Heat Island Analysis

## Business Context
Urban heat islands (UHIs) are metropolitan areas that experience significantly higher temperatures than surrounding rural areas. This analysis identifies heat islands, quantifies temperature variations, and provides actionable recommendations for urban cooling strategies.

## Objective
- Map temperature variations across urban areas
- Identify heat island hotspots
- Analyze correlation with land cover and building density
- Recommend mitigation strategies

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Data Generation
Creating synthetic urban temperature data with realistic patterns

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate grid of urban locations (100x100 points)
grid_size = 100
x = np.linspace(0, 10, grid_size)
y = np.linspace(0, 10, grid_size)
X, Y = np.meshgrid(x, y)

# Simulate temperature with urban core being hotter
# Center of the city at (5, 5)
center_x, center_y = 5, 5
distance_from_center = np.sqrt((X - center_x)**2 + (Y - center_y)**2)

# Base temperature decreases with distance from center
base_temp = 35 - 2 * distance_from_center

# Add secondary heat sources (industrial areas)
industrial_1 = 3 * np.exp(-((X - 3)**2 + (Y - 7)**2) / 2)
industrial_2 = 2.5 * np.exp(-((X - 7)**2 + (Y - 3)**2) / 2)

# Add cooling effects (parks and water bodies)
park_1 = -2 * np.exp(-((X - 2)**2 + (Y - 2)**2) / 1.5)
park_2 = -1.8 * np.exp(-((X - 8)**2 + (Y - 8)**2) / 1.5)

# Combine all effects with random noise
temperature = base_temp + industrial_1 + industrial_2 + park_1 + park_2
temperature += np.random.normal(0, 0.5, temperature.shape)

# Create dataframe
data = pd.DataFrame({
    'longitude': X.flatten(),
    'latitude': Y.flatten(),
    'temperature': temperature.flatten(),
    'building_density': (50 - 5 * distance_from_center.flatten() + np.random.normal(0, 5, X.size)).clip(0, 100),
    'vegetation_index': (distance_from_center.flatten() * 8 + np.random.normal(0, 5, X.size)).clip(0, 100)
})

# Add land cover categories
data['land_cover'] = pd.cut(data['temperature'], 
                              bins=[0, 28, 30, 32, 100],
                              labels=['Green Space', 'Residential', 'Commercial', 'Industrial'])

print(f"Dataset shape: {data.shape}")
data.head(10)

## 2. Exploratory Data Analysis

In [ ]:
# Statistical summary
print("=" * 60)
print("TEMPERATURE STATISTICS")
print("=" * 60)
print(data['temperature'].describe())
print(f"\nTemperature Range: {data['temperature'].max() - data['temperature'].min():.2f}°C")
print(f"Urban Heat Island Intensity: {data['temperature'].max() - data['temperature'].min():.2f}°C")

In [ ]:
# Land cover distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Land cover count
land_cover_counts = data['land_cover'].value_counts()
axes[0].bar(land_cover_counts.index, land_cover_counts.values, color=['#2ecc71', '#3498db', '#f39c12', '#e74c3c'])
axes[0].set_title('Land Cover Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Land Cover Type')
axes[0].set_ylabel('Number of Grid Points')
axes[0].grid(axis='y', alpha=0.3)

# Temperature by land cover
data.boxplot(column='temperature', by='land_cover', ax=axes[1])
axes[1].set_title('Temperature Distribution by Land Cover', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Land Cover Type')
axes[1].set_ylabel('Temperature (°C)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('land_cover_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Heat Island Mapping

In [ ]:
# Create heat map
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Temperature heat map
temp_grid = data.pivot_table(values='temperature', index='latitude', columns='longitude')
im1 = axes[0].contourf(temp_grid.columns, temp_grid.index, temp_grid.values, 
                       levels=20, cmap='RdYlBu_r')
axes[0].set_title('Urban Heat Island Map', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
cbar1 = plt.colorbar(im1, ax=axes[0])
cbar1.set_label('Temperature (°C)', rotation=270, labelpad=20)

# Building density map
density_grid = data.pivot_table(values='building_density', index='latitude', columns='longitude')
im2 = axes[1].contourf(density_grid.columns, density_grid.index, density_grid.values,
                       levels=20, cmap='YlOrRd')
axes[1].set_title('Building Density Map', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
cbar2 = plt.colorbar(im2, ax=axes[1])
cbar2.set_label('Building Density (%)', rotation=270, labelpad=20)

plt.tight_layout()
plt.savefig('heat_island_maps.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation between temperature and urban features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Temperature vs Building Density
axes[0].scatter(data['building_density'], data['temperature'], alpha=0.3, c=data['temperature'], cmap='RdYlBu_r')
z1 = np.polyfit(data['building_density'], data['temperature'], 1)
p1 = np.poly1d(z1)
axes[0].plot(data['building_density'], p1(data['building_density']), "r--", linewidth=2, label=f'y={z1[0]:.3f}x+{z1[1]:.2f}')
axes[0].set_title('Temperature vs Building Density', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Building Density (%)')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Temperature vs Vegetation Index
axes[1].scatter(data['vegetation_index'], data['temperature'], alpha=0.3, c=data['temperature'], cmap='RdYlBu_r')
z2 = np.polyfit(data['vegetation_index'], data['temperature'], 1)
p2 = np.poly1d(z2)
axes[1].plot(data['vegetation_index'], p2(data['vegetation_index']), "r--", linewidth=2, label=f'y={z2[0]:.3f}x+{z2[1]:.2f}')
axes[1].set_title('Temperature vs Vegetation Index', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Vegetation Index')
axes[1].set_ylabel('Temperature (°C)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate correlation coefficients
corr_building = data['temperature'].corr(data['building_density'])
corr_vegetation = data['temperature'].corr(data['vegetation_index'])

print("\n" + "="*60)
print("CORRELATION ANALYSIS")
print("="*60)
print(f"Temperature vs Building Density: r = {corr_building:.3f}")
print(f"Temperature vs Vegetation Index: r = {corr_vegetation:.3f}")

## 5. Heat Island Intensity Classification

In [ ]:
# Classify areas by heat island intensity
mean_temp = data['temperature'].mean()
std_temp = data['temperature'].std()

data['heat_category'] = pd.cut(data['temperature'],
                                bins=[0, mean_temp - std_temp, mean_temp, mean_temp + std_temp, 100],
                                labels=['Cool Zone', 'Moderate', 'Warm Zone', 'Heat Island'])

# Heat island statistics
heat_stats = data.groupby('heat_category').agg({
    'temperature': ['mean', 'count'],
    'building_density': 'mean',
    'vegetation_index': 'mean'
}).round(2)

print("\n" + "="*60)
print("HEAT ISLAND INTENSITY CLASSIFICATION")
print("="*60)
print(heat_stats)

# Visualize heat categories
fig, ax = plt.subplots(figsize=(10, 8))
colors = {'Cool Zone': '#3498db', 'Moderate': '#2ecc71', 'Warm Zone': '#f39c12', 'Heat Island': '#e74c3c'}
for category in data['heat_category'].unique():
    subset = data[data['heat_category'] == category]
    ax.scatter(subset['longitude'], subset['latitude'], 
               c=colors[category], label=category, alpha=0.6, s=20)

ax.set_title('Heat Island Intensity Classification', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(title='Zone Type', loc='best')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('heat_intensity_classification.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Hotspot Analysis

In [ ]:
# Identify top 10 hotspots
hotspots = data.nlargest(100, 'temperature')[['longitude', 'latitude', 'temperature', 'building_density']]

print("\n" + "="*60)
print("TOP 10 HEAT ISLAND HOTSPOTS")
print("="*60)
print(hotspots.head(10))

# Visualize hotspots
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(data['longitude'], data['latitude'], 
                    c=data['temperature'], cmap='RdYlBu_r', alpha=0.5, s=15)
ax.scatter(hotspots['longitude'], hotspots['latitude'], 
          c='red', s=100, marker='*', edgecolors='black', linewidths=1.5,
          label='Critical Hotspots (Top 100)', zorder=5)

ax.set_title('Heat Island Hotspots Identification', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Temperature (°C)', rotation=270, labelpad=20)

plt.tight_layout()
plt.savefig('hotspot_identification.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Key Findings & Recommendations

In [ ]:
# Calculate key metrics
uhi_intensity = data['temperature'].max() - data['temperature'].min()
avg_urban_temp = data['temperature'].mean()
heat_island_area = (data['heat_category'] == 'Heat Island').sum() / len(data) * 100
high_density_areas = (data['building_density'] > 40).sum() / len(data) * 100

print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)
print(f"1. Urban Heat Island Intensity: {uhi_intensity:.2f}°C")
print(f"2. Average Urban Temperature: {avg_urban_temp:.2f}°C")
print(f"3. Heat Island Coverage: {heat_island_area:.1f}% of urban area")
print(f"4. High Building Density Areas: {high_density_areas:.1f}% of urban area")
print(f"5. Correlation with Building Density: {corr_building:.3f} (Strong Positive)")
print(f"6. Correlation with Vegetation: {corr_vegetation:.3f} (Negative)")

print("\n" + "="*70)
print("RECOMMENDATIONS")
print("="*70)
print("\n1. INCREASE GREEN SPACES")
print("   - Plant trees in identified hotspot areas")
print("   - Create urban parks in high-density zones")
print("   - Target: Increase vegetation cover by 20% in heat islands")

print("\n2. COOL ROOFING IMPLEMENTATION")
print("   - Mandate cool/reflective roofs in commercial zones")
print("   - Provide incentives for green roof installations")
print("   - Expected temperature reduction: 2-3°C")

print("\n3. STRATEGIC URBAN PLANNING")
print("   - Limit building density in critical hotspot areas")
print("   - Create ventilation corridors for air circulation")
print("   - Implement cool pavement technologies")

print("\n4. WATER FEATURE INTEGRATION")
print("   - Add fountains and water bodies in heat island zones")
print("   - Restore natural waterways where possible")
print("   - Estimated cooling effect: 1-2°C within 500m radius")

print("\n" + "="*70)

## Conclusion

This analysis successfully identified urban heat islands and their relationships with building density and vegetation cover. The findings provide actionable insights for urban planners to implement cooling strategies and improve livability in metropolitan areas.